# 04 Hierarchical Chunking for 10-K / 10-Q

埋め込み (ベクトル化) を目的とした **filing 構造を活かしたチャンク化** を実装する。
`_helpers.py` には依存せず、本 notebook 内で完結する。

## 戦略

```
Filing (10-K / 10-Q)
  └ Item 階層            ← edgartools の obj() API で取得
      └ 小見出し階層     ← 本 notebook で正規表現/ヒューリスティック検出
          └ 段落 packing ← トークン上限まで段落単位で詰める
```

## 従来手法との比較

| 手法                                   | 境界                | 変化検知の感度                      |
| -------------------------------------- | ------------------- | ----------------------------------- |
| 固定 510 トークン (01_fetch_and_chunk) | 段落途中で分断      | △ 平均で薄まる                      |
| **本 notebook (階層 + 段落 packing)**  | 小見出し + 段落境界 | ◎ 新規見出しが 1 チャンクとして残る |

## ターゲット

- **Item 1A (Risk Factors)**: 小見出しが多く変化検知に最適
- **Item 7 (MD&A)**: Results of Operations / Liquidity 等の構造あり
- **10-Q Part II Item 1A / Part I Item 2**: 同等の構造

## 出力

`data/chunks_hier.parquet` に保存。後段の埋め込み notebook で読み込んで Chamfer 類似度計算に使う。


In [2]:
# Cell 1: imports + EDGAR identity (helpers 非依存)
# HuggingFace キャッシュはデフォルト (~/.cache/huggingface/) を使用する。
# 03-2 で既に gte-Qwen2-1.5B-instruct (6.6 GB) がそこにあるため、再ダウンロードを避ける。
from __future__ import annotations

import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import edgar
from tqdm.auto import tqdm
from IPython.display import display, Markdown, HTML

# データ保存先 (notebook と同階層の data/)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# EDGAR identity を .env から読み込み (リポジトリルートの .env)
env_path = Path("../../.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("EDGAR_IDENTITY="):
            os.environ["EDGAR_IDENTITY"] = (
                line.split("=", 1)[1].strip().strip('"').strip("'")
            )
            break
identity = os.environ.get("EDGAR_IDENTITY")
if not identity:
    raise RuntimeError(
        ".env に EDGAR_IDENTITY='Name email@example.com' を設定してください"
    )


edgar.set_identity(identity)
print("EDGAR identity:", identity)
print("DATA_DIR:", DATA_DIR.resolve())


EDGAR identity: YH-05 youxitiancore@gmail.com
DATA_DIR: /Users/yukihata/Desktop/quants/notebook/FILING_NLP/data


In [3]:
# Cell 2: トークナイザロード
# 後段の埋め込みで使う gte-Qwen2 のトークナイザに合わせる (チャンクサイズが
# embedding 時の実トークン数と一致するため)。
from transformers import AutoTokenizer

TOKENIZER_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)
print("tokenizer:", type(tokenizer).__name__, "vocab:", tokenizer.vocab_size)


tokenizer: Qwen2TokenizerFast vocab: 151643


In [4]:
# Cell 3: サンプル filing 取得
# AAPL / MSFT それぞれ 10-K x 3年 + 10-Q x 4四半期 = 計 14 件。
# 小サンプルで小見出し検出の精度を素早く検証する。
TICKERS = ["AAPL", "MSFT", "GOOGL", "NVDA", "TSLA", "AVGO", "AMAT", "AMZN", "META"]


def _fetch(ticker: str, form: str, limit: int):
    try:
        return (
            ticker,
            form,
            list(edgar.Company(ticker).get_filings(form=form).head(limit)),
        )
    except Exception as e:  # noqa: BLE001
        return ticker, form, e


requests = [(t, "10-K", 3) for t in TICKERS] + [(t, "10-Q", 4) for t in TICKERS]
with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(lambda r: _fetch(*r), requests))

all_filings: list[tuple[str, str, object]] = []
for ticker, form, r in results:
    if isinstance(r, Exception):
        print(f"  ERR {ticker} {form}: {r}")
        continue
    print(f"  {ticker} {form}: {len(r)} 件")
    for f in r:
        all_filings.append((ticker, form, f))

print(f"\ntotal filings: {len(all_filings)}")


  AAPL 10-K: 3 件
  MSFT 10-K: 3 件
  GOOGL 10-K: 3 件
  NVDA 10-K: 3 件
  TSLA 10-K: 3 件
  AVGO 10-K: 3 件
  AMAT 10-K: 3 件
  AMZN 10-K: 3 件
  META 10-K: 3 件
  AAPL 10-Q: 4 件
  MSFT 10-Q: 4 件
  GOOGL 10-Q: 4 件
  NVDA 10-Q: 4 件
  TSLA 10-Q: 4 件
  AVGO 10-Q: 4 件
  AMAT 10-Q: 4 件
  AMZN 10-Q: 4 件
  META 10-Q: 4 件

total filings: 63


In [5]:
# Cell 4: Item 1A (Risk Factors) と Item 7 (MD&A) をセクション単位で抽出
# 10-K / 10-Q とも edgartools 新 API obj.get_item_with_part(part, item) を使用。
# 旧来の .risk_factors 属性や obj['Part II, Item 1A'] subscript は内部で
# legacy parser fallback を呼ぶ可能性があり v6.0 で削除予定のため。
#
# Part/Item の対応:
#   10-K: Part I,  Item 1A → Risk Factors / Part II, Item 7 → MD&A
#   10-Q: Part II, Item 1A → Risk Factors / Part I,  Item 2 → MD&A
#   (MD&A が 10-K と 10-Q で Part が逆になる点に注意)
ITEM_MAP = {
    "10-K": [
        ("item_1a", "Part I", "Item 1A"),
        ("item_7", "Part II", "Item 7"),
    ],
    "10-Q": [
        ("item_1a", "Part II", "Item 1A"),
        ("item_7", "Part I", "Item 2"),
    ],
}

section_rows = []
for ticker, form, f in tqdm(all_filings, desc="extracting sections"):
    try:
        obj = f.obj()
    except Exception as e:  # noqa: BLE001
        print(f"  obj fail: {ticker} {f.accession_number} {e}")
        continue
    fid = str(f.accession_number)
    fdate = pd.Timestamp(str(f.filing_date))
    for key, part, item in ITEM_MAP[form]:
        try:
            text = obj.get_item_with_part(part, item, markdown=False)
        except Exception as e:  # noqa: BLE001
            print(f"  get_item_with_part fail: {ticker} {fid} {part}/{item}: {e}")
            text = None
        if isinstance(text, str) and text:
            section_rows.append(
                {
                    "filing_id": fid,
                    "ticker": ticker,
                    "form": form,
                    "filing_date": fdate,
                    "item_key": key,
                    "text": text,
                    "char_count": len(text),
                }
            )

df_sections = pd.DataFrame(section_rows)
df_sections.to_parquet(DATA_DIR / "item-1a_and_item-7.parquet", index=False)
print(f"\nsections: {len(df_sections)}")
print("expected: 10-K 9x3x2 + 10-Q 9x4x2 = 126")
df_sections.groupby(["ticker", "form", "item_key"]).size()


extracting sections:   0%|          | 0/63 [00:00<?, ?it/s]

TenK falling back to legacy parser for 'Item 1A' (filing: 0001104659-26-053166). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 7' (filing: 0001104659-26-053166). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 1A' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 7' (filing: 00011046


sections: 122
expected: 10-K 9x3x2 + 10-Q 9x4x2 = 126


ticker  form  item_key
AAPL    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
AMAT    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
AMZN    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
AVGO    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
GOOGL   10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
META    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
MSFT    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
NVDA    10-K  item_1a     3
              item_7      3
        10-Q  item_1a     4
              item_7      4
TSLA    10-K  item_1a     1
              item_7      1
        10-Q  item_1a    

## 小見出し検出戦略

edgartools が返す `obj.risk_factors` は HTML タグを除去した plain text のため、
**構造シグナルが lost している**。これを正規表現とヒューリスティックで復元する。

### 検出ルール

候補とする行の条件:

1. 行の長さが 5〜120 文字
2. 末尾が句読点 (`.`, `,`, `;`, `:`) で終わっていない
3. 数字や箇条書き記号 (`•`, `-`, `(a)`, `i.`) で始まらない
4. Title Case 様の見た目 (単語の 60%以上が大文字始まり)

加えて、**強いシグナル**として下記の prefix にマッチする行を優先採用:

- `Risks Related to ...` / `Risks Relating to ...` / `Risks Associated with ...`
- `Strategic Risks` / `Operational Risks` / `Financial Risks` / `Legal Risks` / `Cybersecurity Risks` ...
- `Overview` / `Results of Operations` / `Liquidity and Capital Resources` / `Critical Accounting ...`

### 出力

各セクションを `(subsection_title, subsection_text)` のリストに分解する。
小見出しが 1 つも見つからない場合は、セクション全体を 1 つの subsection として扱う。


In [ ]:
# Cell 5: 小見出し検出 + subsection 分割
# 強シグナルの prefix パターン (大文字小文字区別なし)
STRONG_HEADING_PATTERNS = [
    r"^Risks?\s+Related\s+to\s+",
    r"^Risks?\s+Relating\s+to\s+",
    r"^Risks?\s+Associated\s+with\s+",
    r"^(Strategic|Operational|Financial|Legal|Regulatory|Market|Macroeconomic|"
    r"Industry|Business|General|Cybersecurity|Tax|Intellectual\s+Property|"
    r"Human\s+Capital|Compliance|Environmental|Climate|Geopolitical|"
    r"Legal\s+and\s+Regulatory\s+Compliance)\s+Risks?\b",
    r"^Overview\b",
    r"^Results\s+of\s+Operations\b",
    r"^Liquidity\s+and\s+Capital\s+Resources\b",
    r"^Critical\s+Accounting\s+(Estimates|Policies|Judgments)\b",
    r"^Recent\s+Accounting\s+Pronouncements\b",
    r"^Off-Balance\s+Sheet\s+Arrangements\b",
    r"^Contractual\s+Obligations\b",
    r"^Foreign\s+Currency\b",
    r"^Segment\s+(Results|Operating\s+Performance|Information)\b",
]
STRONG_HEADING_RE = re.compile("|".join(STRONG_HEADING_PATTERNS), re.IGNORECASE)

# ページヘッダ/フッタ等の artifact パターン
# 例: "Apple Inc. | 2025 Form 10-K | 5", "Microsoft Corporation Page 12"
PAGE_ARTIFACT_PATTERNS = [
    r"\|",  # pipe 区切り (page header に頻出)
    r"\bForm\s+(?:10|8|11|20|S)-?\s*[KQABFN]?\b",  # "Form 10-K" 等
    r"\bPage\s+\d+\b",  # "Page 5"
    r"\bAnnual\s+Report\b",
    r"\bQuarterly\s+Report\b",
    r"\bTable\s+of\s+Contents\b",
]
PAGE_ARTIFACT_RE = re.compile("|".join(PAGE_ARTIFACT_PATTERNS), re.IGNORECASE)

# テーブル系プレインテキスト判定用パターン
# edgartools は markdown=False で表構造を失い、数値の羅列が plain text として残る
TABLE_MARKER_RE = re.compile(
    r"(Three|Six|Nine|Twelve)\s+Months\s+Ended|"
    r"Percentage\s*Change|"
    r"\(In\s+millions|"
    r"\(In\s+thousands",
    re.IGNORECASE,
)


def _normalize(s: str) -> str:
    """nbsp (\\xa0) や連続空白を半角空白 1 個に正規化."""
    return re.sub(r"\s+", " ", s.replace("\xa0", " ")).strip()


def _is_page_artifact(line: str) -> bool:
    """ページヘッダ/フッタ等を判定 (見出しでも本文でもない)."""
    return bool(PAGE_ARTIFACT_RE.search(_normalize(line)))


def _is_table_like(text: str) -> bool:
    """財務テーブル系のプレインテキストを判定.

    判定条件: 期間マーカー (Three Months Ended 等) を含み、かつ
    数値比率が高い (デジット文字 / 全文字数 > 5%) または
    連続空白 (3 個以上) が 5 箇所以上ある (表整形の跡).
    """
    if not TABLE_MARKER_RE.search(text):
        return False
    digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
    multi_space_count = len(re.findall(r" {3,}", text))
    return digit_ratio > 0.05 or multi_space_count > 5


def _preprocess(text: str) -> str:
    """edgartools の plain text を段落分割しやすい形に前処理.

    1. bullet (• や ·) で始まる行の前に空行を挿入 → 段落として認識される
    2. 連続改行 (3 個以上) を 2 個に正規化
    """
    # bullet が連続している箇所を段落区切りに昇格
    text = re.sub(r"\n[•·]", "\n\n•", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def _looks_like_heading(line: str) -> bool:
    """弱シグナル: 短い + 句読点で終わらない + Title Case 様 + artifact でない."""
    s = _normalize(line)
    if not (5 <= len(s) <= 120):
        return False
    if s[-1] in ".,;:":
        return False
    if s[0].isdigit() or s[0] in ("•", "-", "*"):
        return False
    # (a), (1), i. ii. iii. のような番号付き
    if re.match(r"^\([a-z]\)|^\([0-9]+\)|^[ivx]+\.", s, re.IGNORECASE):
        return False
    # 末尾が単独の数字 (ページ番号らしさ) なら除外
    if re.search(r"\s\d{1,4}\s*$", s):
        return False
    # ページヘッダ/フッタ系の artifact は見出しではない
    if _is_page_artifact(s):
        return False
    # Title Case 判定: 単語の 60% 以上が大文字始まり
    words = re.findall(r"[A-Za-z]+", s)
    if not words:
        return False
    cap = sum(1 for w in words if w[0].isupper())
    return cap / len(words) >= 0.6


def split_subsections(text: str) -> list[tuple[str, str]]:
    """テキストを (subsection_title, subsection_text) のリストに分割.

    アルゴリズム:
        1. _preprocess で bullet 行を段落区切りに昇格
        2. 連続改行を \\n\\n に正規化してブロック分割
        3. 単一行ブロックが page artifact なら全体スキップ (本文にも残さない)
        4. 各ブロックの先頭行が見出しシグナルなら subsection 境界とする
           - 単一行ブロック: ブロック全体が見出し
           - 複数行ブロック: 先頭行が強シグナルなら見出し + 残りを本文に
    小見出しが 1 つも見つからない場合は [("", text)] を返す.
    """
    text = _preprocess(text.strip())
    blocks = text.split("\n\n")

    items: list[tuple[str, str]] = []
    cur_title = ""
    cur_body: list[str] = []

    def flush() -> None:
        if cur_body:
            items.append((cur_title, "\n\n".join(cur_body).strip()))

    for block in blocks:
        b = block.strip()
        if not b:
            continue
        lines = b.split("\n")
        first_line = lines[0].strip()
        # 単一行ブロック
        if len(lines) == 1:
            # page artifact は本文にも残さず完全スキップ
            if _is_page_artifact(first_line):
                continue
            if STRONG_HEADING_RE.match(first_line) or _looks_like_heading(first_line):
                flush()
                cur_title = first_line
                cur_body = []
                continue
        # 複数行ブロック: 先頭が強シグナルのみ採用 (弱シグナルだと誤検出多)
        elif STRONG_HEADING_RE.match(first_line) and not _is_page_artifact(first_line):
            flush()
            cur_title = first_line
            cur_body = []
            rest = "\n".join(lines[1:]).strip()
            if rest:
                cur_body.append(rest)
            continue
        cur_body.append(b)
    flush()

    if not items:
        return [("", text)]
    return items


# 単体テスト: page artifact が正しく弾かれるか
_test_cases = [
    ("Apple Inc. | 2025 Form 10-K | 5", True),
    ("Microsoft Corporation Page 12", True),
    ("Annual Report 2024", True),
    ("Macroeconomic and Industry Risks", False),
    ("Business Risks", False),
    ("Legal and Regulatory Compliance Risks", False),
    ("Financial Risks", False),
    ("Risks Related to Our Operations", False),
]
for _text, _expected in _test_cases:
    assert _is_page_artifact(_text) == _expected, (
        f"_is_page_artifact({_text!r}) expected {_expected}"
    )
print(f"_is_page_artifact: {len(_test_cases)} test cases passed")

# 単体テスト: _is_table_like
_table_test_cases = [
    # 財務テーブル系 (True)
    ("Three Months Ended March 31, 2026   2025   Percentage Change  100  200", True),
    ("(In millions, except percentages)  Revenue 1,234 2,345", True),
    # 通常文章 (False)
    (
        "The Company is subject to various risks including supply chain disruption.",
        False,
    ),
    ("Risks Related to Our Operations affect our ability to deliver products.", False),
]
for _text, _expected in _table_test_cases:
    assert _is_table_like(_text) == _expected, (
        f"_is_table_like({_text!r}) expected {_expected}, got {_is_table_like(_text)}"
    )
print(f"_is_table_like: {len(_table_test_cases)} test cases passed")

# 単体テスト: _preprocess (bullet 段落化)
_pp_input = "intro:\n•first item\n•second item\n\nnext paragraph"
_pp_expected = "intro:\n\n•first item\n\n•second item\n\nnext paragraph"
assert _preprocess(_pp_input) == _pp_expected, (
    f"_preprocess output mismatch: {_preprocess(_pp_input)!r}"
)
print("_preprocess: 1 test case passed")


In [23]:
def check_item_1a_subsections(ticker):
    # 動作確認 (10-K Item 1A)
    sample_filter = (
        (df_sections["ticker"] == ticker)
        & (df_sections["item_key"] == "item_1a")
        & (df_sections["form"] == "10-K")
    )
    sample = df_sections[sample_filter].sort_values("filing_date").iloc[-1]
    subs = split_subsections(sample["text"])
    print(f"【{ticker}】 {sample['form']} {sample['filing_date'].date()} Item 1A:")
    print(f"  → {len(subs)} subsections detected")
    for i, (title, body) in enumerate(subs):
        label = title if title else "(no title)"
        n_tok = len(tokenizer.encode(body, add_special_tokens=False))
        print(f"  [{i:2d}] {label[:70]:70} | {len(body):7d} chars | {n_tok:6d} tokens")


In [24]:
for ticker in TICKERS:
    check_item_1a_subsections(ticker)


【AAPL】 10-K 2025-10-31 Item 1A:
  → 6 subsections detected
  [ 0] Item 1A.    Risk Factors                                               |     711 chars |    120 tokens
  [ 1] Macroeconomic and Industry Risks                                       |   13498 chars |   2274 tokens
  [ 2] Business Risks                                                         |   25627 chars |   4250 tokens
  [ 3] Legal and Regulatory Compliance Risks                                  |   17909 chars |   3058 tokens
  [ 4] Financial Risks                                                        |    8130 chars |   1412 tokens
  [ 5] General Risks                                                          |    1424 chars |    256 tokens
【MSFT】 10-K 2025-07-30 Item 1A:
  → 8 subsections detected
  [ 0] ITEM 1A. RISK FACTORS                                                  |     269 chars |     45 tokens
  [ 1] STRATEGIC AND COMPETITIVE RISKS                                        |    9208 chars |   1545 tokens
  

In [25]:
# Cell 6: 全セクションに subsection 分割を適用
all_subs_rows = []
for row in df_sections.to_dict("records"):
    subs = split_subsections(row["text"])
    for sub_idx, (title, body) in enumerate(subs):
        all_subs_rows.append(
            {
                "filing_id": row["filing_id"],
                "ticker": row["ticker"],
                "form": row["form"],
                "filing_date": row["filing_date"],
                "item_key": row["item_key"],
                "subsection_idx": sub_idx,
                "subsection_title": title,
                "text": body,
                "char_count": len(body),
            }
        )
df_subs = pd.DataFrame(all_subs_rows)
print(f"total subsections: {len(df_subs)}")

print("\n=== subsections per filing × item ===")
print(df_subs.groupby(["ticker", "form", "item_key"])["subsection_idx"].nunique())

print("\n=== 検出された subsection 数の分布 (filing × item ごと) ===")
print(df_subs.groupby(["filing_id", "item_key"]).size().describe().round(2))

print("\n=== 出現頻度トップ subsection title (空タイトル除く) ===")
top_titles = df_subs[df_subs["subsection_title"] != ""][
    "subsection_title"
].value_counts()
print(top_titles.head(20))


total subsections: 2295

=== subsections per filing × item ===
ticker  form  item_key
AAPL    10-K  item_1a      6
              item_7      36
        10-Q  item_1a      1
              item_7      37
AMAT    10-K  item_1a      7
              item_7      48
        10-Q  item_1a      1
              item_7       1
AMZN    10-K  item_1a     22
              item_7      33
        10-Q  item_1a      1
              item_7       2
AVGO    10-K  item_1a     10
              item_7      32
        10-Q  item_1a     10
              item_7      29
GOOGL   10-K  item_1a      7
              item_7      65
        10-Q  item_1a      1
              item_7      75
META    10-K  item_1a     11
              item_7      53
        10-Q  item_1a     12
              item_7      42
MSFT    10-K  item_1a      9
              item_7      62
        10-Q  item_1a      1
              item_7       1
NVDA    10-K  item_1a      9
              item_7      34
        10-Q  item_1a      1
              i

## 段落 packing

各 subsection をさらに **段落境界を尊重しつつ** トークン上限まで詰める。

### アルゴリズム

1. subsection を `\n\n` で段落リスト化
2. 段落をトークン化し token 数を計算
3. 先頭から累積し、累積 tokens が `MAX_TOKENS` を超える直前で切る
4. **例外**: 単一段落が `MAX_TOKENS` を超える場合は文単位で強制分割

### サイズ設計

- `MAX_TOKENS = 3000`: gte-Qwen2 上限 8192 の約 37%
- 余裕を持たせる理由:
    - last-token pooling は長文で表現が薄まりやすい
    - subsection × 3000 tok で 1 セクションあたり 3〜6 チャンクが目安
    - Chamfer 計算で n × m が小さい方が解釈しやすい


In [ ]:
# Cell 7: 段落 packing 実装 + サンプル確認
MAX_TOKENS = 512


def _sentence_split(text: str) -> list[str]:
    """簡易文分割 (nltk 等を使わず正規表現). 'A. B' のような略語は無視する."""
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    return [p.strip() for p in parts if p.strip()]


def _tok_count(s: str) -> int:
    return len(tokenizer.encode(s, add_special_tokens=False))


def paragraph_pack(text: str, max_tokens: int = MAX_TOKENS) -> list[str]:
    """段落単位で詰め込み. 単一段落が上限超えなら文分割.

    Returns
    -------
    list[str]
        各要素が max_tokens 以下のチャンク. 段落境界を尊重する.
    """
    text = text.strip()
    if not text:
        return []
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks: list[str] = []
    cur: list[str] = []
    cur_tok = 0
    for p in paragraphs:
        n = _tok_count(p)
        # 単一段落が max 超え → 文単位で強制分割
        if n > max_tokens:
            if cur:
                chunks.append("\n\n".join(cur))
                cur, cur_tok = [], 0
            sentences = _sentence_split(p)
            sub_cur: list[str] = []
            sub_tok = 0
            for s in sentences:
                sn = _tok_count(s)
                if sub_tok + sn > max_tokens and sub_cur:
                    chunks.append(" ".join(sub_cur))
                    sub_cur, sub_tok = [], 0
                sub_cur.append(s)
                sub_tok += sn
            if sub_cur:
                chunks.append(" ".join(sub_cur))
            continue
        # 通常: 段落を詰める
        if cur_tok + n > max_tokens and cur:
            chunks.append("\n\n".join(cur))
            cur, cur_tok = [], 0
        cur.append(p)
        cur_tok += n
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks


# 動作確認 (df_subs の先頭サンプル)
sample_sub = df_subs.iloc[0]
sample_chunks = paragraph_pack(sample_sub["text"])
print(
    f"sample: {sample_sub['ticker']} {sample_sub['form']} "
    f"{sample_sub['item_key']} sub[{sample_sub['subsection_idx']}] "
    f"title={sample_sub['subsection_title']!r}"
)
print(f"  char_count={sample_sub['char_count']} → {len(sample_chunks)} chunks")
for i, c in enumerate(sample_chunks):
    n = _tok_count(c)
    print(f"  chunk[{i}] {n:5d} tokens | head: {c[:100]!r}")


sample: AAPL 10-K item_1a sub[0] title='Item 1A.\xa0\xa0\xa0\xa0Risk Factors'
  char_count=711 → 1 chunks
  chunk[0]   120 tokens | head: 'The following summarizes factors that could have a material adverse effect on the Company’s business'


In [ ]:
# Cell 8: 全 subsection に段落 packing を適用 → chunks_hier.parquet 保存
# テーブル系 chunk は _is_table_like で除外 (Chamfer 解析でノイズになるため)
chunk_rows = []
table_skipped = 0
for row in tqdm(df_subs.to_dict("records"), desc="chunking"):
    chunks = paragraph_pack(row["text"])
    for i, c in enumerate(chunks):
        if _is_table_like(c):
            table_skipped += 1
            continue
        chunk_rows.append(
            {
                "filing_id": row["filing_id"],
                "ticker": row["ticker"],
                "form": row["form"],
                "filing_date": row["filing_date"],
                "item_key": row["item_key"],
                "subsection_idx": row["subsection_idx"],
                "subsection_title": row["subsection_title"],
                "chunk_idx": i,
                "text": c,
                "token_count": _tok_count(c),
            }
        )

df_chunks_hier = pd.DataFrame(chunk_rows)
out_path = DATA_DIR / "chunks_hier.parquet"
df_chunks_hier.to_parquet(out_path)
print(f"saved: {out_path.resolve()}")
print(f"total chunks: {len(df_chunks_hier)}")
print(f"table chunks skipped: {table_skipped}")


In [31]:
# Cell 9: 検証統計
print("=== チャンク数: ticker × form × item_key ===")
print(df_chunks_hier.groupby(["ticker", "form", "item_key"]).size())

print("\n=== トークン数分布 ===")
print(df_chunks_hier["token_count"].describe().round(1))

print("\n=== 1 subsection あたりの chunk 数分布 ===")
chunks_per_sub = df_chunks_hier.groupby(
    ["filing_id", "item_key", "subsection_idx"]
).size()
print(chunks_per_sub.describe().round(2))

print("\n=== AAPL 最新 10-K Item 1A の subsection × chunk 一覧 ===")
aapl_filter = (
    (df_chunks_hier["ticker"] == "AAPL")
    & (df_chunks_hier["form"] == "10-K")
    & (df_chunks_hier["item_key"] == "item_1a")
)
aapl_latest_fid = (
    df_chunks_hier[aapl_filter].sort_values("filing_date").iloc[-1]["filing_id"]
)
aapl_view = df_chunks_hier[
    aapl_filter & (df_chunks_hier["filing_id"] == aapl_latest_fid)
][["subsection_idx", "subsection_title", "chunk_idx", "token_count"]]
# print(aapl_view.to_string(index=False))
display(aapl_view)

print("\n=== MSFT 最新 10-K Item 1A の subsection × chunk 一覧 ===")
msft_filter = (
    (df_chunks_hier["ticker"] == "MSFT")
    & (df_chunks_hier["form"] == "10-K")
    & (df_chunks_hier["item_key"] == "item_1a")
)
msft_latest_fid = (
    df_chunks_hier[msft_filter].sort_values("filing_date").iloc[-1]["filing_id"]
)
msft_view = df_chunks_hier[
    msft_filter & (df_chunks_hier["filing_id"] == msft_latest_fid)
][["subsection_idx", "subsection_title", "chunk_idx", "token_count"]]
# print(msft_view.to_string(index=False))
display(msft_view)


=== チャンク数: ticker × form × item_key ===
ticker  form  item_key
AAPL    10-K  item_1a      89
              item_7      104
        10-Q  item_1a      16
              item_7      145
AMAT    10-K  item_1a      95
              item_7      145
        10-Q  item_1a     105
              item_7       85
AMZN    10-K  item_1a      97
              item_7      121
        10-Q  item_1a     103
              item_7       99
AVGO    10-K  item_1a     132
              item_7      118
        10-Q  item_1a     167
              item_7      117
GOOGL   10-K  item_1a     107
              item_7      202
        10-Q  item_1a       6
              item_7      292
META    10-K  item_1a     273
              item_7      162
        10-Q  item_1a     369
              item_7      183
MSFT    10-K  item_1a      96
              item_7      183
        10-Q  item_1a     169
              item_7      136
NVDA    10-K  item_1a     160
              item_7      103
        10-Q  item_1a      67
       

,subsection_idx,subsection_title,chunk_idx,token_count
0,0,Item 1A. Risk Factors,0,120
1,1,Macroeconomic and Industry Risks,0,449
2,1,Macroeconomic and Industry Risks,1,509
3,1,Macroeconomic and Industry Risks,2,142
4,1,Macroeconomic and Industry Risks,3,490
5,1,Macroeconomic and Industry Risks,4,446
6,1,Macroeconomic and Industry Risks,5,238
7,2,Business Risks,0,365
8,2,Business Risks,1,442
9,2,Business Risks,2,228



=== MSFT 最新 10-K Item 1A の subsection × chunk 一覧 ===


,subsection_idx,subsection_title,chunk_idx,token_count
193,0,ITEM 1A. RISK FACTORS,0,45
194,1,STRATEGIC AND COMPETITIVE RISKS,0,376
195,1,STRATEGIC AND COMPETITIVE RISKS,1,513
196,1,STRATEGIC AND COMPETITIVE RISKS,2,457
197,1,STRATEGIC AND COMPETITIVE RISKS,3,199
198,2,RISKS RELATING TO THE EVOLUTION OF OUR BUSINESS,0,294
199,2,RISKS RELATING TO THE EVOLUTION OF OUR BUSINESS,1,414
200,3,"CYBERSECURITY, DATA PRIVACY, AND PLATFORM ABUS...",0,162
201,3,"CYBERSECURITY, DATA PRIVACY, AND PLATFORM ABUS...",1,464
202,3,"CYBERSECURITY, DATA PRIVACY, AND PLATFORM ABUS...",2,372


## 段落 packing 結果の検証 (`MAX_TOKENS = 512`)

9 銘柄すべての chunking 結果を検証する。`paragraph_pack` は **subsection が `MAX_TOKENS` 以下なら 1 chunk として保存する**挙動なので、短い subsection (e.g., META "Summary Risk Factors" 61 tok、AAPL "General Risks" 256 tok) はそのまま残る。

### 確認ポイント

1. **短い subsection 保存**: 合計トークンが `MAX_TOKENS` 以下の subsection は 1 chunk のままか (Cell 12 の `短 subsection (≤512 tok)` 行で確認)
2. **上限遵守**: いずれの chunk も `token_count ≤ MAX_TOKENS` か (Cell 12 の `上限超過 chunk 数` で確認、期待値 0)
3. **段落境界の尊重**: chunk 先頭が文の途中で始まっていないか (Cell 13 のサンプル抜粋で確認)
4. **Cross-section 比較の規模感**: 各 ticker の総 chunk 数バランス (Cell 12 末尾の ticker 別集計)
5. **Time-section の連続性**: 同一 ticker・同一 item でも年度ごとに subsection 数が大きく変動しないか (Cell 11 で目視)

### Cell 11 の表示ルール

`[keep]` = 単一 chunk のまま保存された subsection (短い)
`[ Nch]` = N 個の chunk に分割された subsection (長い)


In [3]:
MAX_TOKENS = 512
df_chunks_hier = pd.read_parquet(DATA_DIR / "chunks_hier.parquet")


In [ ]:
amzn_item7 = df_chunks_hier.loc[
    (df_chunks_hier["ticker"] == "AMZN")
    & (df_chunks_hier["item_key"] == "item_7")
    & (df_chunks_hier["subsection_title"] == "")
]
display(amzn_item7)


,filing_id,ticker,form,filing_date,item_key,subsection_idx,subsection_title,chunk_idx,text,token_count
3764,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,0,Item 2. Management’s Discussion and Analysis o...,478
3765,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,1,Critical Accounting Estimates\nThe preparation...,483
3766,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,2,"Tax laws, regulations, administrative practice...",93
3767,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,3,"change due to economic, political, and other c...",373
3768,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,4,Three Months Ended Twelve Months Ended ...,512
3769,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,5,Cash provided by (used in) investing activitie...,327
3770,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,6,Cash provided by (used in) financing activitie...,485
3771,0001018724-26-000014,AMZN,10-Q,2026-04-30,item_7,0,,7,"As of December 31, 2025 and March 31, 2026, re...",441
3812,0001018724-25-000123,AMZN,10-Q,2025-10-31,item_7,0,,0,Item 2. Management’s Discussion and Analysis o...,512
3813,0001018724-25-000123,AMZN,10-Q,2025-10-31,item_7,0,,1,Critical accounting estimates are those estima...,492


In [5]:
# Cell 11: 全 ticker の最新 10-K Item 1A について subsection → chunks の対応を表示
def show_chunks_per_ticker(
    ticker: str, form: str = "10-K", item_key: str = "item_1a"
) -> None:
    """指定 ticker の最新 filing について subsection 単位の chunk breakdown を表示."""

    f = (
        (df_chunks_hier["ticker"] == ticker)
        & (df_chunks_hier["form"] == form)
        & (df_chunks_hier["item_key"] == item_key)
    )
    if not f.any():
        print(f"【{ticker}】 {form} {item_key}: データなし")
        return
    latest_fid = df_chunks_hier[f].sort_values("filing_date").iloc[-1]["filing_id"]
    view = df_chunks_hier[f & (df_chunks_hier["filing_id"] == latest_fid)]
    fdate = view["filing_date"].iloc[0].date()
    n_subs = view["subsection_idx"].nunique()
    n_chunks = len(view)
    total_tok = int(view["token_count"].sum())
    print(
        f"【{ticker}】 {form} {fdate} {item_key}: "
        f"{n_subs} subsections → {n_chunks} chunks ({total_tok:,} tokens)"
    )
    by_sub = (
        view.groupby(["subsection_idx", "subsection_title"])
        .agg(
            n_chunks=("chunk_idx", "size"),
            total_tokens=("token_count", "sum"),
            max_tokens=("token_count", "max"),
        )
        .reset_index()
        .sort_values("subsection_idx")
    )
    for _, r in by_sub.iterrows():
        title = (r["subsection_title"] or "(no title)")[:55]
        # 短 subsection (1 chunk のまま保存) を [keep] でマーク
        tag = "[keep]" if r["n_chunks"] == 1 else f"[{r['n_chunks']:>2d}ch]"
        print(
            f"  {tag} sub[{r['subsection_idx']:2d}] {title:55} | "
            f"{r['total_tokens']:6d} tok (max chunk {r['max_tokens']:4d})"
        )


for ticker in TICKERS:
    show_chunks_per_ticker(ticker)
    print()


【AAPL】 10-K 2025-10-31 item_1a: 6 subsections → 30 chunks (11,370 tokens)
  [keep] sub[ 0] Item 1A.    Risk Factors                                |    120 tok (max chunk  120)
  [ 6ch] sub[ 1] Macroeconomic and Industry Risks                        |   2274 tok (max chunk  509)
  [10ch] sub[ 2] Business Risks                                          |   4250 tok (max chunk  512)
  [ 8ch] sub[ 3] Legal and Regulatory Compliance Risks                   |   3058 tok (max chunk  472)
  [ 4ch] sub[ 4] Financial Risks                                         |   1412 tok (max chunk  452)
  [keep] sub[ 5] General Risks                                           |    256 tok (max chunk  256)

【MSFT】 10-K 2025-07-30 item_1a: 8 subsections → 30 chunks (11,201 tokens)
  [keep] sub[ 0] ITEM 1A. RISK FACTORS                                   |     45 tok (max chunk   45)
  [ 4ch] sub[ 1] STRATEGIC AND COMPETITIVE RISKS                         |   1545 tok (max chunk  513)
  [ 2ch] sub[ 2] RISKS RELA

In [10]:
# Cell 12: 健全性チェック (短 subsection 保存・上限超過・packing 効率)

print("=== 設定 ===")
print(f"  MAX_TOKENS:                     {MAX_TOKENS}")
print(f"  総 chunk 数:                     {len(df_chunks_hier):,}")
print(f"  対象 ticker 数:                  {df_chunks_hier['ticker'].nunique()}")
print(f"  対象 filing 数:                  {df_chunks_hier['filing_id'].nunique()}")

print("\n=== 上限・下限の健全性 ===")
over = int((df_chunks_hier["token_count"] > MAX_TOKENS).sum())
empty = int((df_chunks_hier["token_count"] == 0).sum())
tiny = int((df_chunks_hier["token_count"] < 50).sum())
print(f"  上限超過 chunk 数:               {over}  (期待値: 0)")
print(f"  空 chunk 数:                     {empty}  (期待値: 0)")
print(f"  極小 chunk (<50 tok):            {tiny}")

print("\n=== 短い subsection が保存されているか ===")
chunks_per_sub = (
    df_chunks_hier.groupby(["filing_id", "item_key", "subsection_idx"])
    .agg(n_chunks=("chunk_idx", "size"), total_tokens=("token_count", "sum"))
    .reset_index()
)
n_total = len(chunks_per_sub)
n_single = int((chunks_per_sub["n_chunks"] == 1).sum())
n_split = int((chunks_per_sub["n_chunks"] > 1).sum())
print(
    f"  1 chunk で済んだ subsection:    {n_single} / {n_total}  ({n_single / n_total:.1%})"
)
print(
    f"  分割された subsection:           {n_split} / {n_total}  ({n_split / n_total:.1%})"
)
print(f"  最大 chunk 数 (単一 subsection): {chunks_per_sub['n_chunks'].max()}")

# 短い subsection (合計トークン ≤ MAX_TOKENS) が確実に 1 chunk になっているか
short_subs = chunks_per_sub[chunks_per_sub["total_tokens"] <= MAX_TOKENS]
short_split = int((short_subs["n_chunks"] > 1).sum())
print(
    f"  短 subsection (≤{MAX_TOKENS} tok): "
    f"{len(short_subs)} 個, うち分割されたもの = {short_split} (期待値: 0)"
)

print("\n=== Token 数分布 ===")
print(df_chunks_hier["token_count"].describe().round(1).to_string())

print("\n=== Packing 効率 (token_count / MAX_TOKENS) ===")
fill_ratio = df_chunks_hier["token_count"] / MAX_TOKENS
print(f"  平均充填率: {fill_ratio.mean():.1%}")
print(f"  中央値:    {fill_ratio.median():.1%}")
print(f"  25-75%:    {fill_ratio.quantile(0.25):.1%} - {fill_ratio.quantile(0.75):.1%}")

print("\n=== Ticker 別 総 chunk 数 (cross-section の規模感) ===")
ticker_summary = (
    df_chunks_hier.groupby("ticker")
    .agg(
        n_chunks=("chunk_idx", "size"),
        n_filings=("filing_id", "nunique"),
        total_tokens=("token_count", "sum"),
    )
    .sort_values("n_chunks", ascending=False)
)
print(ticker_summary.to_string())


=== 設定 ===
  MAX_TOKENS:                     512
  総 chunk 数:                     4,490
  対象 ticker 数:                  9
  対象 filing 数:                  61

=== 上限・下限の健全性 ===
  上限超過 chunk 数:               40  (期待値: 0)
  空 chunk 数:                     0  (期待値: 0)
  極小 chunk (<50 tok):            653

=== 短い subsection が保存されているか ===
  1 chunk で済んだ subsection:    1993 / 2295  (86.8%)
  分割された subsection:           302 / 2295  (13.2%)
  最大 chunk 数 (単一 subsection): 44
  短 subsection (≤512 tok): 1992 個, うち分割されたもの = 0 (期待値: 0)

=== Token 数分布 ===
count    4490.0
mean      275.2
std       178.5
min         2.0
25%        97.0
50%       287.0
75%       457.0
max       840.0

=== Packing 効率 (token_count / MAX_TOKENS) ===
  平均充填率: 53.7%
  中央値:    56.1%
  25-75%:    18.9% - 89.3%

=== Ticker 別 総 chunk 数 (cross-section の規模感) ===
        n_chunks  n_filings  total_tokens
ticker                                   
META         987          7        306624
GOOGL        607          7        106844
MSFT 

In [11]:
# Cell 13: 各 ticker からサンプル chunk のテキスト抜粋
# 短 subsection (1 chunk のまま保存) と長 subsection (分割) の両方を確認する
print("=== 各 ticker の最新 10-K Item 1A サンプル chunk ===\n")
for ticker in TICKERS:
    f = (
        (df_chunks_hier["ticker"] == ticker)
        & (df_chunks_hier["form"] == "10-K")
        & (df_chunks_hier["item_key"] == "item_1a")
    )
    if not f.any():
        continue
    latest_fid = df_chunks_hier[f].sort_values("filing_date").iloc[-1]["filing_id"]
    view = df_chunks_hier[f & (df_chunks_hier["filing_id"] == latest_fid)].reset_index(
        drop=True
    )

    # 最短 chunk (短 subsection が保存された証拠)
    shortest = view.loc[view["token_count"].idxmin()]
    # 最大 chunk (packing 上限近くまで詰まったもの)
    longest = view.loc[view["token_count"].idxmax()]

    print(f"━━━ 【{ticker}】 chunks: {len(view)} ━━━")
    print(
        f"  [最短] sub[{shortest['subsection_idx']}] "
        f"{(shortest['subsection_title'] or '(no title)')!r} "
        f"chunk[{shortest['chunk_idx']}] ({shortest['token_count']} tok)"
    )
    print(f"    {shortest['text']!r}")
    print(
        f"  [最大] sub[{longest['subsection_idx']}] "
        f"{(longest['subsection_title'] or '(no title)')!r} "
        f"chunk[{longest['chunk_idx']}] ({longest['token_count']} tok)"
    )
    print(f"    {longest['text']!r}")
    print()


=== 各 ticker の最新 10-K Item 1A サンプル chunk ===

━━━ 【AAPL】 chunks: 30 ━━━
  [最短] sub[3] 'Legal and Regulatory Compliance Risks' chunk[7] (100 tok)
    'The Company is also subject to new and changing laws and regulations regarding online safety, including enhanced protections for minors and mandatory age verification requirements. These laws and regulations can increase regulatory risks by requiring complex compliance measures and significant modifications to the Company’s products, services and operations, and may lead to operational disruptions, heightened privacy and data security risks, increased costs and potential liability and fines, all of which can have a material adverse impact on the Company’s business, financial condition, results of operations and stock price.'
  [最大] sub[2] 'Business Risks' chunk[3] (512 tok)
    'The Company offers complex hardware and software products and services that can be affected by design and manufacturing defects. Sophisticated operating system so

In [12]:
# Cell 14: 上限超過 chunk の調査
# paragraph_pack は単一文 > MAX_TOKENS のケースで分割しきれず、その文を 1 chunk
# として吐き出す。Cell 9 で max_token_count が MAX_TOKENS を超えていたら以下を確認:
#   1. 規模 (全体の何 %)
#   2. 分布 (どの ticker / item / subsection に集中しているか)
#   3. テキスト内容 (legal enumeration か / 表残骸か / 略語誤検出か)
#   4. 分割ヒント (;, :, • の有無で clause-split が効くか判定)
oversize = df_chunks_hier[df_chunks_hier["token_count"] > MAX_TOKENS]
n_total = len(df_chunks_hier)
n_over = len(oversize)

print("=== 上限超過 chunk の規模 ===")
print(f"  MAX_TOKENS:         {MAX_TOKENS}")
print(f"  上限超過 chunk 数:   {n_over} / {n_total} ({n_over / n_total:.1%})")

if n_over == 0:
    print("\n上限超過 chunk なし。対策不要。")
else:
    print("\n=== 超過分の token 分布 ===")
    print(oversize["token_count"].describe().round(0).to_string())

    print("\n=== ticker × form × item_key 別の発生数 ===")
    print(oversize.groupby(["ticker", "form", "item_key"]).size().to_string())

    print("\n=== subsection_title 別の発生数 (top 10) ===")
    print(oversize["subsection_title"].value_counts().head(10).to_string())

    # 分割ヒント: 全 oversize chunk で句読点・記号の数を集計
    # → clause-split (;:) が効くか、bullet (•) で分けられるかの判断材料
    print("\n=== 分割ヒント (oversize chunk の文字統計) ===")
    stats = pd.DataFrame(
        {
            "semicolons": oversize["text"].str.count(";"),
            "colons": oversize["text"].str.count(":"),
            "bullets": oversize["text"].str.count("•"),
            "newlines": oversize["text"].str.count("\n"),
            "periods": oversize["text"].str.count(r"\."),
        }
    )
    print(stats.describe().round(1).to_string())
    print(
        f"  ; or : を含む chunk 数:  "
        f"{((stats['semicolons'] + stats['colons']) > 0).sum()} / {n_over}"
    )

    print("\n=== 上位 5 chunk のテキスト先頭 (token_count 降順) ===")
    top5 = oversize.nlargest(5, "token_count")
    for i, (_, row) in enumerate(top5.iterrows()):
        title = (row["subsection_title"] or "(no title)")[:40]
        print(
            f"\n--- [{i + 1}] {row['ticker']} {row['form']} "
            f"sub='{title}' ({row['token_count']} tok) ---"
        )
        # 改行を空白に置換して表示しやすく
        text_preview = row["text"][:500].replace("\n", " ⏎ ")
        print(f"  {text_preview!r}")


=== 上限超過 chunk の規模 ===
  MAX_TOKENS:         512
  上限超過 chunk 数:   40 / 4490 (0.9%)

=== 超過分の token 分布 ===
count     40.0
mean     587.0
std       85.0
min      513.0
25%      514.0
50%      582.0
75%      627.0
max      840.0

=== ticker × form × item_key 別の発生数 ===
ticker  form  item_key
AMAT    10-K  item_1a     5
        10-Q  item_1a     8
              item_7      5
AMZN    10-Q  item_1a     4
              item_7      4
AVGO    10-K  item_1a     1
        10-Q  item_1a     1
META    10-K  item_7      1
        10-Q  item_7      2
MSFT    10-K  item_1a     1
              item_7      1
        10-Q  item_7      4
NVDA    10-Q  item_7      1
TSLA    10-K  item_1a     1
              item_7      1

=== subsection_title 別の発生数 (top 10) ===
subsection_title
                                                     21
Business and Industry Risks                           4
Results of Operations                                 4
Foreign Exchange Impact on Revenue                    3
Risks Re

In [13]:
no_title = df_chunks_hier[df_chunks_hier["subsection_title"] == ""]
table_marker_pattern = re.compile(
    r"(Three|Six|Nine|Twelve)\s+Months\s+Ended|Percentage\s*Change|\(In\s+millions",
    re.IGNORECASE,
)
print(f"=== (no title) subsection の chunk 分析 ===")
print(f"  総数: {len(no_title)} / {len(df_chunks_hier)}")
table_like = no_title["text"].apply(lambda t: bool(table_marker_pattern.search(t)))
print(f"  テーブル系: {table_like.sum()} / {len(no_title)}")
print(f"\n  ticker × form 別:")
print(no_title[table_like].groupby(["ticker", "form", "item_key"]).size().to_string())


=== (no title) subsection の chunk 分析 ===
  総数: 645 / 4490
  テーブル系: 115 / 645

  ticker × form 別:
ticker  form  item_key
AMAT    10-Q  item_7      47
AMZN    10-Q  item_7      14
MSFT    10-Q  item_7      54
